## Govdata Catalog Fetch

Ziel ist es den gesamten Govdata Metadaten Katalog zu fetchen und möglichst platzsparend zu speichern um lokal darauf zu zu greifen.

In [5]:
import json
import sys
from pathlib import Path
from typing import Any, Dict

import requests


CKAN_BASE_URL = "https://ckan.govdata.de"


# Client functions (without class)
def create_session() -> requests.Session:
    """Create a configured requests session."""
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": "govdata-ckan-client/0.1",
            "Accept": "application/json",
        }
    )
    return session


def fetch_dataset_metadata(dataset_id: str, timeout: int = 30) -> Dict[str, Any]:
    """
    Holt die vollständigen Metadaten eines Datensatzes über package_show.
    """
    session = create_session()
    package_show_url = f"{CKAN_BASE_URL}/api/3/action/package_show"
    
    response = session.get(
        package_show_url,
        params={"id": dataset_id},
        timeout=timeout,
    )
    response.raise_for_status()

    payload = response.json()

    if not payload.get("success", False):
        raise RuntimeError(
            f"CKAN API meldet success=false für Dataset-ID {dataset_id}: {payload}"
        )

    result = payload.get("result")
    if result is None:
        raise RuntimeError(
            f"CKAN API hat kein 'result' für Dataset-ID {dataset_id} geliefert."
        )

    return result


# Helper functions
def save_json(data: Any, output_path: Path) -> None:
    output_path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


# Test
result = fetch_dataset_metadata("832a64df-be7f-4fb8-8fd4-7190049e5bd4")
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "author": "Bundesamt für Justiz",
  "author_email": "poststelle@bfj.bund.de",
  "creator_user_id": "d210c2e1-faa9-4218-a42f-75bf3ead66df",
  "id": "832a64df-be7f-4fb8-8fd4-7190049e5bd4",
  "isopen": false,
  "license_id": null,
  "license_title": null,
  "maintainer": null,
  "maintainer_email": null,
  "metadata_created": "2026-03-13T12:49:17.308369",
  "metadata_modified": "2026-03-13T12:49:17.308373",
  "name": "gzr-daten-zur-schwarzarbeit-2024-naturliche-personen-bugeldentscheidungen-nach-8-abs-1-nr-2-i-v-m-8-",
  "notes": "Mit der Übersicht „GZR-Daten zur Schwarzarbeit 2024“ legt das Bundesamt für Justiz das Ergebnis von Auswertungen der im Jahr 2024 in die Teilregister über natürliche Personen sowie juristische Personen und Personenvereinigungen des Gewerbezentralregisters eingetragenen Bußgeldentscheidungen wegen Ordnungswidrigkeiten im Bereich der Schwarzarbeit vor. Der Datensatz weist alle Bußgeldentscheidungen gegen natürliche Personen aus, die im Auswertungsjahr nach § 8

fetch entire ckan govdata catalog

In [29]:
import requests

BASE_URL = "https://www.govdata.de/ckan/api/3/action/package_list"

def fetch(url):
    print(f"Fetching: {url}")
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

data = fetch(BASE_URL)
packages = data.get("result", [])
print(f"Total packages: {len(packages)}")
print("First 5 package IDs: \n" + "\n".join(packages[:5]))

Fetching: https://www.govdata.de/ckan/api/3/action/package_list
Total packages: 151185
First 5 package IDs: 
__
000-einwohner
001-bookholzberg-urschrift
001-bookholzberg-urschrift-1
003-ganderkesee-urschrift


ich will 50 random id's und von den selektierten ID's die Metadaten fetchen.

In [32]:
import random

random_packages = random.sample(packages, 10)

metadata_list = []

for pkg_id in random_packages:

    metadata = fetch(f"https://www.govdata.de/ckan/api/3/action/package_show?id={pkg_id}")
    metadata_list.append(metadata)


for meta in metadata_list:
    print("-----------------------------------------------------------------------")
    print(json.dumps(meta, ensure_ascii=False, indent=2))

Fetching: https://www.govdata.de/ckan/api/3/action/package_show?id=bebauungsplan-petershagen-dorfkern-und-angrenzende-gebiete-10-anderung-gemeinde-petershagen-egg-1
Fetching: https://www.govdata.de/ckan/api/3/action/package_show?id=innenbereichssatzung-lichterfelde-gemeinde-schorfheide-ot-lichterfelde-wfs
Fetching: https://www.govdata.de/ckan/api/3/action/package_show?id=eichenau-lebendgeborene
Fetching: https://www.govdata.de/ckan/api/3/action/package_show?id=bebauungsplan-fuhlsbuttel-13-hamburg
Fetching: https://www.govdata.de/ckan/api/3/action/package_show?id=das-baugewerbe-in-hamburg-august-2018
Fetching: https://www.govdata.de/ckan/api/3/action/package_show?id=bebauungsplan-hinter-der-alten-schmiede
Fetching: https://www.govdata.de/ckan/api/3/action/package_show?id=topographische-karte-1-25-000-ausgabe-staat-1981-1989-n-33-138-c-a-eisenhuttenstadt-west
Fetching: https://www.govdata.de/ckan/api/3/action/package_show?id=flachennutzungsplane-der-verwaltungsgemeinschaft-seenplatte
Fet

another try to fetch some random files out of the catalog with rdf or ttl format

In [2]:
import pathlib
import random
import sys
import time
from typing import Any, Dict, List, Optional

import requests

API_BASE = "https://govdata.de/ckan/api/3/action/package_search"

# CKAN DCAT dataset endpoint pattern documented by ckanext-dcat:
#   /dataset/{dataset-id}.{format}
#
# GovData has historically exposed CKAN under /ckan for the API, while the RDF
# dataset route is typically at the site root. To avoid guessing wrong, the
# script tries both common route variants.
DATASET_URL_PATTERNS = [
    "https://www.govdata.de/dataset/{name}.{fmt}",
    "https://www.govdata.de/ckan/dataset/{name}.{fmt}",
    "https://govdata.de/dataset/{name}.{fmt}",
    "https://govdata.de/ckan/dataset/{name}.{fmt}",
]

FORMAT = "rdf"   # use "rdf" or "xml" for RDF/XML
SAMPLE_SIZE = 10
OUT_DIR = pathlib.Path("govdata_random_rdf")
TIMEOUT = 60
USER_AGENT = "govdata-rdf-random-sampler/1.0"


def package_search(session: requests.Session, rows: int, start: int) -> Dict[str, Any]:
    resp = session.get(
        API_BASE,
        params={"rows": rows, "start": start},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    payload = resp.json()

    if not payload.get("success"):
        raise RuntimeError(f"CKAN API returned success=false: {payload}")

    return payload["result"]


def get_catalog_count(session: requests.Session) -> int:
    result = package_search(session, rows=0, start=0)
    count = result.get("count")
    if not isinstance(count, int) or count <= 0:
        raise RuntimeError(f"Unexpected catalog count: {count!r}")
    return count


def sample_offsets(count: int, n: int) -> List[int]:
    n = min(n, count)
    return random.sample(range(count), n)


def get_dataset_name_at_offset(session: requests.Session, offset: int) -> str:
    result = package_search(session, rows=1, start=offset)
    records = result.get("results", [])
    if not records:
        raise RuntimeError(f"No dataset returned for offset {offset}")

    record = records[0]
    name = record.get("name")
    if not name:
        raise RuntimeError(f"Dataset at offset {offset} has no 'name' field")

    return name


def fetch_dataset_rdf(
    session: requests.Session,
    dataset_name: str,
    fmt: str,
) -> Optional[bytes]:
    accept_map = {
        "ttl": "text/turtle",
        "rdf": "application/rdf+xml",
        "xml": "application/rdf+xml",
        "n3": "text/n3",
        "jsonld": "application/ld+json",
    }

    headers = {"Accept": accept_map.get(fmt, "*/*")}

    for pattern in DATASET_URL_PATTERNS:
        url = pattern.format(name=dataset_name, fmt=fmt)
        try:
            resp = session.get(url, headers=headers, timeout=TIMEOUT, allow_redirects=True)
        except requests.RequestException:
            continue

        if resp.status_code != 200:
            continue

        content_type = (resp.headers.get("Content-Type") or "").lower()
        body = resp.content[:500].lower()

        if fmt == "ttl":
            if b"@prefix" in body or "text/turtle" in content_type:
                return resp.content
        else:
            if b"<rdf:rdf" in body or "application/rdf+xml" in content_type or "xml" in content_type:
                return resp.content

    return None


def main() -> int:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    try:
        total = get_catalog_count(session)
        offsets = sample_offsets(total, SAMPLE_SIZE)

        print(f"Catalog size: {total}")
        print(f"Sampling offsets: {offsets}")
        print()

        saved = 0
        failed = []

        for i, offset in enumerate(offsets, start=1):
            try:
                name = get_dataset_name_at_offset(session, offset)
                content = fetch_dataset_rdf(session, name, FORMAT)

                if content is None:
                    failed.append((offset, name, "RDF endpoint not resolved"))
                    print(f"[{i}] FAIL  offset={offset}  name={name}  reason=RDF endpoint not resolved")
                    continue

                outfile = OUT_DIR / f"{name}.{FORMAT}"
                outfile.write_bytes(content)
                saved += 1

                print(f"[{i}] OK    offset={offset}  name={name}  saved={outfile}")
                time.sleep(0.2)

            except Exception as e:
                failed.append((offset, None, str(e)))
                print(f"[{i}] FAIL  offset={offset}  reason={e}")

        print()
        print(f"Saved {saved} file(s) in {OUT_DIR.resolve()}")

        if failed:
            print("Failures:")
            for item in failed:
                print(f"  {item}")

        return 0 if saved > 0 else 1

    except Exception as e:
        print(f"ERROR: {e}", file=sys.stderr)
        return 1


if __name__ == "__main__":
    raise SystemExit(main())

Catalog size: 151116
Sampling offsets: [38927, 113962, 120523, 109171, 65520, 132496, 60632, 42557, 145380, 121957]

[1] OK    offset=38927  name=bebauungsplan-04-2017-lilienweg-gemeinde-michendorf-ot-michendorf-wfs  saved=govdata_random_rdf/bebauungsplan-04-2017-lilienweg-gemeinde-michendorf-ot-michendorf-wfs.rdf
[2] OK    offset=113962  name=wohngebaudebestand-nach-anzahl-der-wohnungen-in-wesselburener-deichhausen  saved=govdata_random_rdf/wohngebaudebestand-nach-anzahl-der-wohnungen-in-wesselburener-deichhausen.rdf
[3] OK    offset=120523  name=corona-daten-rendsburg-eckernforde36b4e  saved=govdata_random_rdf/corona-daten-rendsburg-eckernforde36b4e.rdf
[4] OK    offset=109171  name=bromberger-strassea6948  saved=govdata_random_rdf/bromberger-strassea6948.rdf
[5] OK    offset=65520  name=bebauungsplan-4-w-urschrift-1  saved=govdata_random_rdf/bebauungsplan-4-w-urschrift-1.rdf
[6] OK    offset=132496  name=betriebe-im-verarbeitenden-gewerbe-nach-wirtschaftsabteilungen-in-gross-ronnau4

SystemExit: 0

This worked it pulled random metadata sets in ttl format